In [2]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

data = pd.read_csv('exoTrain.csv')

X = data.drop('LABEL', axis=1).values
y = data['LABEL'].values - 1  # make labels 0,1

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# convert AFTER split
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)

# reshape for LSTM
X_train = X_train.unsqueeze(1)
X_val   = X_val.unsqueeze(1)


train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class CNN_LSTM(nn.Module):
    def __init__(self):
        super().__init__()

        # CNN part
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        # LSTM part
        self.lstm = nn.LSTM(
            input_size=32,   # from CNN output channels
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )

        # Fully connected
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        x = self.conv(x)  # CNN


        # reshape for LSTM
        x = x.permute(0, 2, 1)   #(batch, seq_len, features)

        # LSTM
        out, _ = self.lstm(x)
    
        out = out[:, -1, :]# take last timestep

        out = self.fc(out)    # classifier


        return out

model = CNN_LSTM()

class_counts = torch.bincount(y_train)
class_weights = 1.0 / class_counts.float()
sample_weights = class_weights[y_train]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler
)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        outputs = model(xb)
        _, preds = torch.max(outputs, 1)

        all_preds.append(preds)
        all_labels.append(yb)

# combine batches
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print("Total evaluated samples:", len(all_preds))

accuracy = (all_preds == all_labels).float().mean()
print("Validation Accuracy:", accuracy.item())

print(confusion_matrix(all_labels.numpy(), all_preds.numpy()))
print(classification_report(all_labels.numpy(), all_preds.numpy()))

Epoch 1, Train Loss: 10.7602
Epoch 2, Train Loss: 3.3212
Epoch 3, Train Loss: 1.8307
Epoch 4, Train Loss: 0.9262
Epoch 5, Train Loss: 0.6409
Epoch 6, Train Loss: 0.7533
Epoch 7, Train Loss: 0.5536
Epoch 8, Train Loss: 1.7937
Epoch 9, Train Loss: 0.6989
Epoch 10, Train Loss: 0.7059
Epoch 11, Train Loss: 0.5269
Epoch 12, Train Loss: 0.5495
Epoch 13, Train Loss: 0.3557
Epoch 14, Train Loss: 0.5212
Epoch 15, Train Loss: 0.5575
Epoch 16, Train Loss: 0.3825
Epoch 17, Train Loss: 0.3996
Epoch 18, Train Loss: 0.4037
Epoch 19, Train Loss: 0.8817
Epoch 20, Train Loss: 0.5543
Total evaluated samples: 1018
Validation Accuracy: 0.8870334029197693
[[900 111]
 [  4   3]]
              precision    recall  f1-score   support

           0       1.00      0.89      0.94      1011
           1       0.03      0.43      0.05         7

    accuracy                           0.89      1018
   macro avg       0.51      0.66      0.49      1018
weighted avg       0.99      0.89      0.93      1018



In [3]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

data = pd.read_csv('exoTrain.csv')

X = data.drop('LABEL', axis=1).values
y = data['LABEL'].values - 1  # make labels 0,1

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# convert AFTER split
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)

# reshape for LSTM
X_train = X_train.unsqueeze(1)
X_val   = X_val.unsqueeze(1)


train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class CNN_LSTM(nn.Module):
    def __init__(self):
        super().__init__()

        # CNN part
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        # LSTM part
        self.lstm = nn.LSTM(
            input_size=32,   # from CNN output channels
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )

        # Fully connected
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        x = self.conv(x)  # CNN


        # reshape for LSTM
        x = x.permute(0, 2, 1)   #(batch, seq_len, features)

        # LSTM
        out, _ = self.lstm(x)
    
        out = out[:, -1, :]# take last timestep

        out = self.fc(out)    # classifier


        return out

model = CNN_LSTM()

class_counts = torch.bincount(y_train)
class_weights = 1.0 / class_counts.float()
sample_weights = class_weights[y_train]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler
)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        outputs = model(xb)
        probs = torch.softmax(outputs, dim=1)
        preds = (probs[:,1] > 0.3).long()

        all_preds.append(preds)
        all_labels.append(yb)

# combine batches
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print("Total evaluated samples:", len(all_preds))

accuracy = (all_preds == all_labels).float().mean()
print("Validation Accuracy:", accuracy.item())

print(confusion_matrix(all_labels.numpy(), all_preds.numpy()))
print(classification_report(all_labels.numpy(), all_preds.numpy())) #threshold tuning 

Epoch 1, Train Loss: 13.3689
Epoch 2, Train Loss: 4.0515
Epoch 3, Train Loss: 2.3421
Epoch 4, Train Loss: 1.7743
Epoch 5, Train Loss: 1.8542
Epoch 6, Train Loss: 3.7822
Epoch 7, Train Loss: 1.1897
Epoch 8, Train Loss: 1.0967
Epoch 9, Train Loss: 1.5182
Epoch 10, Train Loss: 1.1070
Epoch 11, Train Loss: 1.1013
Epoch 12, Train Loss: 4.8138
Epoch 13, Train Loss: 1.4579
Epoch 14, Train Loss: 1.2119
Epoch 15, Train Loss: 0.8923
Epoch 16, Train Loss: 0.7402
Epoch 17, Train Loss: 0.8044
Epoch 18, Train Loss: 0.5771
Epoch 19, Train Loss: 0.4549
Epoch 20, Train Loss: 0.7556
Total evaluated samples: 1018
Validation Accuracy: 0.8300589323043823
[[842 169]
 [  4   3]]
              precision    recall  f1-score   support

           0       1.00      0.83      0.91      1011
           1       0.02      0.43      0.03         7

    accuracy                           0.83      1018
   macro avg       0.51      0.63      0.47      1018
weighted avg       0.99      0.83      0.90      1018



In [ ]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

data = pd.read_csv('exoTrain.csv')

X = data.drop('LABEL', axis=1).values
y = data['LABEL'].values - 1  # make labels 0,1

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# convert AFTER split
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)

# reshape for LSTM
X_train = X_train.unsqueeze(1)
X_val   = X_val.unsqueeze(1)


train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class CNN_LSTM(nn.Module):
    def __init__(self):
        super().__init__()

        # CNN part
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=5),#####here
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        # LSTM part
        self.lstm = nn.LSTM(
            input_size=128,   # from CNN output channels
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )

        # Fully connected
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        x = self.conv(x)  # CNN


        # reshape for LSTM
        x = x.permute(0, 2, 1)   #(batch, seq_len, features)

        # LSTM
        out, _ = self.lstm(x)
    
        out = out[:, -1, :]# take last timestep

        out = self.fc(out)    # classifier


        return out

model = CNN_LSTM()

class_counts = torch.bincount(y_train)
class_weights = 1.0 / class_counts.float()
sample_weights = class_weights[y_train]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler
)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        outputs = model(xb)
        probs = torch.softmax(outputs, dim=1)
        preds = (probs[:,1] > 0.3).long()

        all_preds.append(preds)
        all_labels.append(yb)

# combine batches
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print("Total evaluated samples:", len(all_preds))

accuracy = (all_preds == all_labels).float().mean()
print("Validation Accuracy:", accuracy.item())

print(confusion_matrix(all_labels.numpy(), all_preds.numpy()))
print(classification_report(all_labels.numpy(), all_preds.numpy())) #threshold tuning + adding stronger feature maps 

Epoch 1, Train Loss: 12.1688
Epoch 2, Train Loss: 4.6036
Epoch 3, Train Loss: 1.9269
Epoch 4, Train Loss: 1.9829
Epoch 5, Train Loss: 1.4332
Epoch 6, Train Loss: 0.9749
Epoch 7, Train Loss: 1.2177
Epoch 8, Train Loss: 0.6460
Epoch 9, Train Loss: 0.4366
Epoch 10, Train Loss: 1.0513
Epoch 11, Train Loss: 0.4760
Epoch 12, Train Loss: 0.3672
Epoch 13, Train Loss: 0.2772
Epoch 14, Train Loss: 0.5318
Epoch 15, Train Loss: 1.0373
Epoch 16, Train Loss: 0.5627
Epoch 17, Train Loss: 0.9031
Epoch 18, Train Loss: 0.4249
Epoch 19, Train Loss: 0.9256
Epoch 20, Train Loss: 0.2645
Total evaluated samples: 1018
Validation Accuracy: 0.9597249627113342
[[974  37]
 [  4   3]]
              precision    recall  f1-score   support

           0       1.00      0.96      0.98      1011
           1       0.07      0.43      0.13         7

    accuracy                           0.96      1018
   macro avg       0.54      0.70      0.55      1018
weighted avg       0.99      0.96      0.97      1018

